In [1]:
import sys
sys.path.append('../')
from src.core.scraper.app import ScrapingUtils
from src.core.scraper.processor import ImagesProcessor
from src.core.scraper.utils import download_images
from src.utils.map_models_name import map_and_validate_model, get_brand_from_url
images_processor = ImagesProcessor()
scraper_utils = ScrapingUtils()

In [2]:
# with open(r'C:\Users\JTRUJILLO\Documents\Galgo\Scripts\Otros\scrape_websites_refactorv2\src\data\html_temp.html', 'r', encoding='utf-8') as f:
#     html_content = f.read()

In [3]:
# from bs4 import BeautifulSoup
# html = html_content

# soup = BeautifulSoup(html, "html.parser")
# links = [f"https://www.auteco.com.co{a.get('href')}" for a in soup.find_all("a")]

In [4]:
# links = ['https://www.auteco.com.co/moto-victory-one-mp/p',
#  'https://www.auteco.com.co/apache-rtr-160-4v-xconnect-abs/p',
#  'https://www.auteco.com.co/apache-rtr-200-4v-xconnect-abs-1/p',
#  'https://www.auteco.com.co/moto-tvs-apache-rtr-160-4v-fi/p',
#  'https://www.auteco.com.co/apache-200-fi-abs-edicion-especial-1/p',
#  'https://www.auteco.com.co/moto-victory-venom-14/p',
#  'https://www.auteco.com.co/moto-victory-venom-18/p',
#  'https://www.auteco.com.co/apache-160-fi-abs-edicion-especial-1/p',
#  'https://www.auteco.com.co/moto-tvs-apache-rtr-310/p',
#  'https://www.auteco.com.co/moto-tvs-apache-200-4v-xc-fi-abs/p',
#  'https://www.auteco.com.co/kawasaki-kle-500/p',
#  'https://www.auteco.com.co/kawasaki-ninja-zx-4rr/p',
#  'https://www.auteco.com.co/moto-zontes-368g/p',
#  'https://www.auteco.com.co/kawasaki-z500/p',
#  'https://www.auteco.com.co/kawasaki-ninja-zx-6r/p',
#  'https://www.auteco.com.co/moto-kawasaki-versys-300-abs/p',
#  'https://www.auteco.com.co/moto-kawasaki-versys-650/p',
#  'https://www.auteco.com.co/kawasaki-z900/p',
#  'https://www.auteco.com.co/moto-benelli-trk-251/p',
#  'https://www.auteco.com.co/moto-zontes-703-rr/p',
#  'https://www.auteco.com.co/moto-benelli-trk-502-x/p',
#  'https://www.auteco.com.co/moto-zontes-155-u/p']

In [9]:
# links = ["https://grupouma.com/colombia/motos/dominar/dominar-400-volcano/"]
# url = "https://grupouma.com/colombia/motos/pulsar/pulsar-ns200-fi-abs-dc/"

urls = ["https://www.auteco.com.co/moto-tvs-raider-125-fi/p"]
BRAND = "TVS"

extract_images = True
extract_technical_specs = True

extract_model_data = False

### Verificar nombramiento

In [10]:
for url in urls:
    try:
        print("Ejecutando extracción de model_data")
        model_data, content = images_processor.get_model_data(url=url)
        if model_data:
            print("Hay model_data")
        else:
            print("No hay model_data")
    except Exception as e:
        print(f"Error extrayendo model_data: {e}")
        model_data = None

    try:
        marca = get_brand_from_url(url)
        model_data = map_and_validate_model(model_data, marca)
    except ValueError as e:
        print(f"\n[MAPEO] {e}")

Ejecutando extracción de model_data
Hay model_data
Archivo de mapeo cargado: C:\Users\JTRUJILLO\Documents\Galgo\Scripts\Otros\scrape_websites_refactorv2\src\data\json\name_mapping\auteco_tvs_mapeo_nombres.json
Archivo de mapeo guardado: C:\Users\JTRUJILLO\Documents\Galgo\Scripts\Otros\scrape_websites_refactorv2\src\data\json\name_mapping\auteco_tvs_mapeo_nombres.json

[MAPEO] Modelo 'RAIDER 125 FI' no encontrado en el mapeo para marca 'auteco_tvs'.
Se agregó al archivo: C:\Users\JTRUJILLO\Documents\Galgo\Scripts\Otros\scrape_websites_refactorv2\src\data\json\name_mapping\auteco_tvs_mapeo_nombres.json
Por favor, edita el archivo y agrega el nombre correcto del marketplace, luego vuelve a ejecutar.


## Ejecutar flujo

In [7]:
for url in urls:
    try:
        print(f"\nProcesando URL: {url}")

        if extract_model_data:
            try:
                print("Ejecutando extracción de model_data")
                model_data, content = images_processor.get_model_data(url=url)
                if model_data:
                    print("Hay model_data")
                else:
                    print("No hay model_data")
            except Exception as e:
                print(f"Error extrayendo model_data: {e}")
                model_data = None
                continue  # Si no hay model_data, no podemos continuar

            # Mapear nombre del modelo al nombre correcto del marketplace
            # Si no está mapeado o tiene valor vacío, se detiene el flujo con instrucciones
            try:
                marca = get_brand_from_url(url)
                model_data = map_and_validate_model(model_data, marca)
            except ValueError as e:
                print(f"\n[MAPEO] {e}")
                continue  # Detener procesamiento de esta URL hasta completar el mapeo

        if extract_images:
            try:
                print("Ejecutando extracción de imágenes")
                images = images_processor.get_images_from_website(url=url)
                print("Imageness")
                print(images)
                if images:
                    print(f"Total de imágenes: {len(images)}")
                else:
                    print("No hubo imágenes")
                # Descargar imágenes
                if model_data and hasattr(model_data, 'model'):
                    # Normalizar nombre para evitar problemas con espacios y caracteres especiales
                    safe_model_name = model_data.model.replace(" ", "_").replace("/", "_")
                    safe_model_name = model_data.model.replace(BRAND, "").strip()
                    safe_brand_name = BRAND.replace(" ", "_")
                    base_name = f"{safe_brand_name}_{safe_model_name}"
                    output_dir = f"../src/data/images/{safe_brand_name}_{safe_model_name}"

                    print(f"Intentando descargar {len(images)} imágenes...")
                    results = download_images(images, base_name, output_dir)

                    # Mostrar resultados
                    successful = sum(1 for r in results if r.get("ok", False))
                    failed = len(results) - successful
                    print(f"Descarga completada: {successful} exitosas, {failed} fallidas")

                    if failed > 0:
                        print("Errores encontrados:")
                        for r in results:
                            if not r.get("ok", False):
                                print(f"  - {r.get('url', 'N/A')}: {r.get('error', 'Error desconocido')}")
                else:
                    print("No se puede descargar: model_data o model no disponible")
            except Exception as e:
                print(f"Error extrayendo imágenes: {e}")
                import traceback
                traceback.print_exc()
                images = None

        if extract_technical_specs:
            try:
                print("Ejecutando extracción de fichas técnicas")
                technical_specs = images_processor.get_technical_specs(url=url)
                if technical_specs:
                    print("Hay ficha técnica")
                    print(technical_specs)
                else:
                    print("No hay ficha técnica")
                # Guardar ficha técnica
                if model_data and hasattr(model_data, 'model'):
                    with open(f"../src/data/technical_specs/{BRAND} {model_data.model}.html", "w", encoding="utf-8") as f:
                        f.write(str(technical_specs))
                    print(f"Archivo HTML guardado como '{BRAND} {model_data.model}.html'")
            except Exception as e:
                print(f"Error extrayendo fichas técnicas: {e}")
                technical_specs = None
    except Exception as e:
        print(f"Error general en la URL {url}: {e}")
        continue


Procesando URL: https://www.auteco.com.co/moto-tvs-raider-125/p
Ejecutando extracción de imágenes
url https://www.auteco.com.co/moto-tvs-raider-125/p
website: auteco tvs
Tipo de contenido: Images
https://media.autecomobility.com/recursos/marcas/tvs/actualizacion-periodica-banners/cotizar-credito/banner-cotizar-credito-tvs-raider-125.webp
https://auteco.vtexassets.com/arquivos/DIVISOR-1024x49.webp
https://auteco.vtexassets.com/arquivos/ids/1506637-30px-30px?width=30px&height=30px&aspect=true
https://auteco.vtexassets.com/arquivos/ids/1506649-30px-30px?width=30px&height=30px&aspect=true
https://media.autecomobility.com/recursos/marcas/tvs/raider-125/interna-de-producto/Imagen_Fondo_Texto_detalle_1_TVS.webp
Marca detectada: TVS en Imagen_Fondo_Texto_detalle_1_TVS.webp
URL base tradicional: https://media.autecomobility.com/recursos/marcas/tvs/raider-125/interna-de-producto/
Marca detectada: TVS
No se encontró la imagen con la extensión por defecto, se intenta con la extensión alternativa


In [8]:
# extract_model_data = False
# extract_images = False
# extract_technical_specs = True

# try:
#     print(f"\nProcesando URL: {url}")

#     if extract_model_data:
#         try:
#             print("Ejecutando extracción de model_data")
#             model_data, content = images_processor.get_model_data(url=url)
#             if model_data:
#                 print("Hay model_data")
#             else:
#                 print("No hay model_data")
#         except Exception as e:
#             print(f"Error extrayendo model_data: {e}")
#             model_data = None

#         # Mapear nombre del modelo al nombre correcto del marketplace
#         # Si no está mapeado o tiene valor vacío, se detiene el flujo con instrucciones
#         try:
#             marca = get_brand_from_url(url)
#             model_data = map_and_validate_model(model_data, marca)
#         except ValueError as e:
#             print(f"\n[MAPEO] {e}")

#     if extract_images:
#         try:
#             print("Ejecutando extracción de imágenes")
#             images = images_processor.get_images_from_website(url=url)
#             print("Imageness")
#             print(images)
#             if images:
#                 print(f"Total de imágenes: {len(images)}")
#             else:
#                 print("No hubo imágenes")
#             # Descargar imágenes
#             if model_data and hasattr(model_data, 'model'):
#                 # Normalizar nombre para evitar problemas con espacios y caracteres especiales
#                 safe_model_name = model_data.model.replace(" ", "_").replace("/", "_")
#                 safe_brand_name = BRAND.replace(" ", "_")
#                 base_name = f"{safe_brand_name}_{safe_model_name}"
#                 output_dir = f"../src/data/images/{safe_brand_name}_{safe_model_name}"

#                 print(f"Intentando descargar {len(images)} imágenes...")
#                 results = download_images(images, base_name, output_dir)

#                 # Mostrar resultados
#                 successful = sum(1 for r in results if r.get("ok", False))
#                 failed = len(results) - successful
#                 print(f"Descarga completada: {successful} exitosas, {failed} fallidas")

#                 if failed > 0:
#                     print("Errores encontrados:")
#                     for r in results:
#                         if not r.get("ok", False):
#                             print(f"  - {r.get('url', 'N/A')}: {r.get('error', 'Error desconocido')}")
#             else:
#                 print("No se puede descargar: model_data o model no disponible")
#         except Exception as e:
#             print(f"Error extrayendo imágenes: {e}")
#             import traceback
#             traceback.print_exc()
#             images = None

#     if extract_technical_specs:
#         try:
#             print("Ejecutando extracción de fichas técnicas")
#             technical_specs = images_processor.get_technical_specs(url=url)
#             if technical_specs:
#                 print("Hay ficha técnica")
#             else:
#                 print("No hay ficha técnica")
#             # Guardar ficha técnica
#             if model_data and hasattr(model_data, 'model'):
#                 with open(f"../src/data/technical_specs/{BRAND} {model_data.model}.html", "w", encoding="utf-8") as f:
#                     f.write(str(technical_specs))
#                 print(f"Archivo HTML guardado como '{BRAND} {model_data.model}.html'")
#         except Exception as e:
#             print(f"Error extrayendo fichas técnicas: {e}")
#             technical_specs = None
# except Exception as e:
#     print(f"Error general en la URL {url}: {e}")